In [ ]:
from pyspark.sql import DataFrame
from typing import Optional, List, Tuple, Dict
from pyspark.sql.functions import col
from pyspark.sql.utils import AnalysisException
def validate_df_cols_standard_nonstandard(df: DataFrame, col_name: Optional[int], expected: int, isStandard: bool = True)-> None:
    """
    Verifies standard concepts are hydrated with non 0 concept ids and non standard concept are 0
    Arguments:
        df: DataFrame - DataFrame to validate
        col_name: str - Column name to validate
        expected: int - Expected count of non 0 concept ids
        isStandard: bool - True if standard concept, False if non standard concept
    Returns:
        None, raises ValidationFailureException if validation fails
    """
    if isStandard:
        col_count = df.filter(col('standard_concept')=='S').collect()[0].__getitem__(col_name)
        if col_count <= expected:
            raise ValidationFailureException('Standard concept should have non 0 concept_id rows')
    if not isStandard:
        if df.filter(col('standard_concept') != 'S').count() != expected:
            raise ValidationFailureException('Non Standard concept count should be 0')

In [ ]:

def validate_df(df: DataFrame, 
                columns: Optional[List[str]] = None, 
                n_cols: Optional[int] = None, 
                check_null_values: Optional[List[str]] = None,
                ) -> Tuple[bool, str]:

    """
    Validates a DataFrame based on specified criteria.

    Parameters:
    - df (pd.DataFrame): The DataFrame to be validated.
    - columns (list, optional): List of column names that should be present in the DataFrame.
    - n_cols (int, optional): Number of expected columns in the DataFrame.
    - check_null_values (bool, optional): Check for the presence of null values in the DataFrame.

    Returns:
    - tuple: (bool, str) indicating success or failure, and an optional description of the problem.
    """


    # Validate number of columns
    if n_cols is not None and len(df.columns) != n_cols:
        error_message = f"Error: Expected {n_cols}columns, but found {len(df.columns)}columns."
        raise ValidationFailureException(error_message)

    # Validate columns
    if columns is not None:
            if not set(columns).issubset(df.columns):
                missing_columns = set(columns) - set(df.columns)
                error_message = f"Error: Missing columns: {missing_columns}."
                raise ValidationFailureException(error_message) 

    # Validate null values  ONLY SPECIFIC COLUMNS
    for value in check_null_values:
        if value and df.filter(df[value].isNull()).count() != 0:
            error_message = "DataFrame contains null values."
            raise  ValidationFailureException(error_message)
    return (True, "Dataframe has passed all the validations")

In [ ]:
def validate_df_col_count(actual: int, expected: int)-> None:
    """validates column value is greater than the expected target

    Arguments:
        actual: int - actual value
        expected: int - target value
    Returns:
        None, raises ValidationFailureException if the actual  value is lesser than the target 
    """
    if actual <= expected:
        raise ValidationFailureException('Value is lower than the target expected')

In [ ]:
def validate_df_col_range(actual: int, desired_range: Tuple[int,int])-> None:
    """Validates column value lies between the range.

    Arguments:
        actual: int - Column value to validate
        desired_range: Tuple[int,int] - Tuple of of lower bound and upper bound value
    Returns:
        None, raises ValidationFailureException if the actual value is out of bounds
    """
    if actual<= desired_range[0] or actual >= desired_range[1]:
        raise ValidationFailureException('Provided value is out of bounds')

In [ ]:
def verify_transformed_data_quality(silver_omop_config: dict, silver_lakehouse: str, assert_dfs: bool)-> None:
    """Verfies the omop transformed data is ingested completely

    Arguments:
        silver_omop_config: dict - A dictionary of silver_omop_config: fhir source table name, source table filtering condition, omop target table name and expected count
        silver_lakehouse: str - Silver lakehouse name
        assert_dfs: flag if true checks both the dfs have equal row counts

    Returns:
        None, raises ValidationFailureException if the row counts does not match for the silver and omop tables or AnalysisException if any of the SQL queries fail
    """
    for key, value in silver_omop_config.items():
        source_table = key
        for (omop_target_table, condition, expected_count) in value:   
            # Check if expected_count is None only when assert_dfs is False
            if not assert_dfs and expected_count is None:
                raise ConfigurationInvalidException(f"Invalid configuration: `expected_count` is None for the table {source_table} -> {omop_target_table} when `assert_dfs` is False.")
            try:
                if condition:        
                    silver_df = spark.sql(f"SELECT * FROM {silver_lakehouse}.{source_table}").filter(condition)
                else:
                    silver_df = spark.sql(f"SELECT * FROM {silver_lakehouse}.{source_table}")

                omop_df = spark.sql(f"SELECT * FROM {omop_target_table}")
                source_count = silver_df.count()
                target_count = omop_df.count()

                if assert_dfs:
                    assert source_count == target_count
                elif expected_count: 
                    source_count = expected_count
                    assert source_count == target_count
            except AnalysisException as e:
                raise AnalysisException(f"SQL query failed for table {source_table} or {omop_target_table}. Details: {str(e)}")
            except AssertionError as e:
                raise ValidationFailureException(f"Silver and OMOP lakehouse row counts does not match for the tables: {source_table}-{source_count}<->{omop_target_table}-{target_count}")

In [ ]:
def omop_concept_validation(omop_tables_concept_pairs: Dict[str, list[int]])-> None:
    """Test verifies that 100% of all the sample data shipped have the associated concept codes and concept_ids populated in OMOP.

     Parameters:
        - df (pd.DataFrame): The DataFrame to be validated.
        - omop_tables_concept_pairs: This is a dictionary where:
            - Keys (str): Each key represents the table name in OMOP lakehouse.
            - Values (list[int]): Each value is a list of integers representing concept IDs associated with the corresponding OMOP table.

    Returns:
        None, raises ValidationFailureException if any of the concept_ids are not mapped resulting in null values
    """
    for table, concepts in omop_tables_concept_pairs.items():
        concept_id_cols = ',' .join(f"{concept}" for concept in concepts)
        print(concept_id_cols)
        query = f"SELECT {concept_id_cols} FROM {table};"
        df = spark.sql(query)
        if df.isEmpty():
            continue
        for concept in concepts:
            concept_id_count = df.filter(col(concept) != 0).count() # when the mapping is corrupted the concept_ids will default to 0
            print(f"Column '{concept}': in {table} Count is : {concept_id_count}")
            if(concept_id_count == 0):
                raise ValidationFailureException(f"Column '{concept}': are populated with default 0 values in {table} : {concept_id_count}")

In [ ]:
# Test suite validation exception
def ValidationFailureException(exception: str):
    raise Exception(exception)

def ConfigurationInvalidException(exception: str):
    raise Exception(exception)